# Postprocessing method comparison

Compares three postprocessing strategies on the **training set** (LOAO CV) for XGBoost,
and on the full train split for TCN (if inference manifest is present).

| Method | Input | Bias |
|--------|-------|------|
| `min_duration_filter` | hard labels | late (absorbs into preceding class) |
| `majority_vote_filter` | hard labels | symmetric |
| `smooth_probs_argmax` | soft probabilities | symmetric |

**Metrics**: signed timing error (bias), MAE timing error, timing std, macro F1.

In [ ]:
import joblib
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from experiments.gait_detection.config import ExperimentConfig
from src.gait.detection.metrics import per_class_f1, timing_error_full
from src.gait.detection.postprocess import (
    derive_events,
    min_duration_filter,
    majority_vote_filter,
    smooth_probs_argmax,
)
from src.gait.gait_data.dataset import load_dataset, loao_splits, train_test_split
from src.gait.image.dataset import load_image_dataset

# ── config ────────────────────────────────────────────────────────────────────
TCN_MANIFEST       = "data/output/gait/pose_data/stage6/manifest.json"
IMG_TCN_MANIFEST   = "data/output/gait/image/stage6/manifest.json"
XGB_PARAMS_PATH    = "data/output/gait/pose_data/stage3/best_params.json"
IMG_XGB_PARAMS_PATH = "data/output/gait/image/stage1/results.json"

IMG_FEATURES_DIR = "data/output/gait/image_features"
VIDEO_INPUT_DIR  = "data/input/optojump"

WINDOWS    = [3, 5, 7, 9, 11]
EVENT_KEYS = ["left_landing", "left_takeoff", "right_landing", "right_takeoff"]

In [ ]:
cfg = ExperimentConfig()
all_records = load_dataset(cfg.annotations_csv, fps=cfg.fps)
train_records, test_records, test_athletes = train_test_split(all_records)

n_athletes = len({r.athlete for r in train_records})
print(f"Train : {len(train_records)} records  ({n_athletes} athletes)")
print(f"Test  : {len(test_records)} records   (excluded: {test_athletes})")

# Image records have a different label clipping window (from frame 0) compared
# to pose records (from first YOLO-detected frame), so load them separately.
img_records_by_path = {}
if os.path.exists(IMG_FEATURES_DIR):
    img_all = load_image_dataset(cfg.annotations_csv, IMG_FEATURES_DIR, VIDEO_INPUT_DIR)
    img_records_by_path = {r.video_path: r for r in img_all}
    print(f"Image : {len(img_all)} records loaded")
else:
    print(f"Image features not found ({IMG_FEATURES_DIR}) — image TCN evaluation will be skipped.")

## Helper functions

In [ ]:
def apply_postprocessing(method: str, data: np.ndarray, window: int) -> np.ndarray:
    """Dispatch to the right postprocessing function.

    `data` is a 1-D label array for hard-label methods,
    or a (T, n_classes) probability array for smooth_probs_argmax.
    """
    if method == "min_duration_filter":
        return min_duration_filter(data, min_frames=window)
    if method == "majority_vote_filter":
        return majority_vote_filter(data, window=window)
    if method == "smooth_probs_argmax":
        return smooth_probs_argmax(data, window=window)
    raise ValueError(f"Unknown postprocessing method: {method!r}")


PROB_METHODS = frozenset({"smooth_probs_argmax"})

# All (method, window) combinations to evaluate
METHODS_CONFIG = (
    [("min_duration_filter",  w) for w in WINDOWS] +
    [("majority_vote_filter", w) for w in WINDOWS] +
    [("smooth_probs_argmax",  w) for w in WINDOWS]
)


def _mean_safe(lst):
    vals = [v for v in lst if not np.isnan(v)]
    return float(np.mean(vals)) if vals else float("nan")


def eval_fold_metrics(val_records, preds, fps, n_classes, class_names):
    """Compute F1 and per-event timing for one fold."""
    all_true, all_pred = [], []
    timing = {k: [] for k in EVENT_KEYS}

    for rec, pred in zip(val_records, preds):
        all_true.append(rec.labels)
        all_pred.append(pred)
        gt_ev   = derive_events(rec.labels, fps)
        pred_ev = derive_events(pred, fps)
        for key in EVENT_KEYS:
            timing[key].append(timing_error_full(pred_ev[key], gt_ev[key], fps))

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    return {
        "f1":     per_class_f1(y_true, y_pred, n_classes, class_names),
        "timing": {
            key: {
                "ms":        _mean_safe([d["ms"]        for d in timing[key]]),
                "signed_ms": _mean_safe([d["signed_ms"] for d in timing[key]]),
            }
            for key in EVENT_KEYS
        },
    }

---
## XGBoost — LOAO CV

For each fold: train on N-1 athletes, run `predict_proba` on the held-out athlete.  
Then apply every (method, window) combination and compute metrics.

In [ ]:
with open(XGB_PARAMS_PATH) as f:
    xgb_meta = json.load(f)
XGB_PARAMS    = xgb_meta["best_params"]
feature_idx   = xgb_meta.get("feature_idx")  # list[int] or None

def _select(features, idx):
    return features if idx is None else features[:, idx]

# ── LOAO: collect raw probs + labels per fold ─────────────────────────────────
xgb_fold_data = []

for fold_i, (fold_train, fold_val, athlete) in enumerate(loao_splits(train_records), 1):
    X_tr = np.vstack([_select(r.features, feature_idx) for r in fold_train])
    y_tr = np.concatenate([r.labels for r in fold_train])

    clf = xgb.XGBClassifier(
        **XGB_PARAMS,
        eval_metric="mlogloss", tree_method="hist", device="cpu", verbosity=0,
    )
    clf.fit(X_tr, y_tr)

    fold_probs  = [clf.predict_proba(_select(r.features, feature_idx)) for r in fold_val]
    fold_labels = [p.argmax(axis=1).astype(np.int64) for p in fold_probs]

    xgb_fold_data.append({
        "athlete":     athlete,
        "val_records": fold_val,
        "raw_labels":  fold_labels,
        "probs":       fold_probs,
    })
    print(f"  [{fold_i:2d}/{n_athletes}] held-out: {athlete}  ({len(fold_val)} videos)")

print("Done.")

In [ ]:
xgb_results = {}  # (method, window) -> list[fold_result]

for method, window in METHODS_CONFIG:
    needs_probs = method in PROB_METHODS
    fold_results = []

    for fold in xgb_fold_data:
        preds = [
            apply_postprocessing(method, fold["probs"][i] if needs_probs else fold["raw_labels"][i], window)
            for i in range(len(fold["val_records"]))
        ]
        fold_results.append(
            eval_fold_metrics(fold["val_records"], preds, cfg.fps, cfg.n_classes, cfg.class_names)
            | {"athlete": fold["athlete"]}
        )

    xgb_results[(method, window)] = fold_results

print(f"Computed {len(xgb_results)} (method, window) combinations for XGBoost.")

---
## Image XGBoost — inference on train set

Loads the pre-trained image XGBoost model and applies it to all train records.
Same in-sample approach as the TCN section — LOAO would be prohibitively slow
at 512–1280 CNN feature dimensions with up to 800 trees per fold.

In [ ]:
img_xgb_results = None
IMG_XGB_MODEL_PATH = "data/output/gait/image/stage1/checkpoints/xgboost_best.pkl"

if not os.path.exists(IMG_XGB_MODEL_PATH):
    print(f"Image XGBoost model not found ({IMG_XGB_MODEL_PATH}).\nRun img_baselines to enable comparison.")
elif not img_records_by_path:
    print("Image records not loaded — skipping image XGBoost evaluation.")
else:
    img_xgb_clf = joblib.load(IMG_XGB_MODEL_PATH)

    img_xgb_train_recs = [
        img_records_by_path[r.video_path]
        for r in train_records
        if r.video_path in img_records_by_path
    ]
    img_xgb_train_probs  = [img_xgb_clf.predict_proba(r.features) for r in img_xgb_train_recs]
    img_xgb_train_labels = [p.argmax(axis=1).astype(np.int64) for p in img_xgb_train_probs]

    print(f"Loaded image XGBoost model  ({len(img_xgb_train_recs)}/{len(train_records)} train records matched)")

    single_fold_img_xgb = [{
        "athlete":     "all_train",
        "val_records": img_xgb_train_recs,
        "raw_labels":  img_xgb_train_labels,
        "probs":       img_xgb_train_probs,
    }]

    img_xgb_results = {}
    for method, window in METHODS_CONFIG:
        needs_probs = method in PROB_METHODS
        preds = [
            apply_postprocessing(method, single_fold_img_xgb[0]["probs"][i] if needs_probs
                                 else single_fold_img_xgb[0]["raw_labels"][i], window)
            for i in range(len(img_xgb_train_recs))
        ]
        result = eval_fold_metrics(
            img_xgb_train_recs, preds, cfg.fps, cfg.n_classes, cfg.class_names
        ) | {"athlete": "all_train"}
        img_xgb_results[(method, window)] = [result]

    print(f"Computed {len(img_xgb_results)} (method, window) combinations for Image XGBoost.")

---
## TCN — inference manifest

Loads raw softmax probabilities from `tcn_inference.py` output.  
**Note:** the TCN model here was trained on all train athletes (not LOAO-style),
so metrics reflect in-sample postprocessing effects rather than held-out generalisation.
This is sufficient for comparing *which postprocessing method* reduces bias.

In [ ]:
tcn_results = None  # will be populated if manifest is found

if not os.path.exists(TCN_MANIFEST):
    print(f"Manifest not found ({TCN_MANIFEST}).\nRun 'make infer' to enable TCN comparison.")
else:
    with open(TCN_MANIFEST) as f:
        manifest = json.load(f)

    manifest_idx = {r["video_path"]: r for r in manifest["records"]}
    tcn_train_recs   = [r for r in train_records if r.video_path in manifest_idx
                        and manifest_idx[r.video_path]["split"] == "train"]
    tcn_train_probs  = [np.load(manifest_idx[r.video_path]["probs_path"]) for r in tcn_train_recs]
    tcn_train_labels = [p.argmax(axis=1).astype(np.int64) for p in tcn_train_probs]

    print(f"Loaded manifest — trial #{manifest['trial_id']}  "
          f"({len(tcn_train_recs)}/{len(train_records)} train records matched)")

    # Treat all train records as a single "fold"
    single_fold = [{
        "athlete":     "all_train",
        "val_records": tcn_train_recs,
        "raw_labels":  tcn_train_labels,
        "probs":       tcn_train_probs,
    }]

    tcn_results = {}
    for method, window in METHODS_CONFIG:
        needs_probs = method in PROB_METHODS
        preds = [
            apply_postprocessing(method, single_fold[0]["probs"][i] if needs_probs
                                 else single_fold[0]["raw_labels"][i], window)
            for i in range(len(tcn_train_recs))
        ]
        result = eval_fold_metrics(
            tcn_train_recs, preds, cfg.fps, cfg.n_classes, cfg.class_names
        ) | {"athlete": "all_train"}
        tcn_results[(method, window)] = [result]

    print(f"Computed {len(tcn_results)} (method, window) combinations for TCN.")

---
## Image TCN — inference manifest

Same postprocessing comparison for the image-based TCN.
The model was trained on all train athletes (not LOAO-style), same as the pose TCN.

In [ ]:
img_tcn_results = None

if not os.path.exists(IMG_TCN_MANIFEST):
    print(f"Image TCN manifest not found ({IMG_TCN_MANIFEST}).\nRun img_tcn_infer to enable comparison.")
else:
    with open(IMG_TCN_MANIFEST) as f:
        img_manifest = json.load(f)

    img_manifest_idx = {r["video_path"]: r for r in img_manifest["records"]}

    # Use image records for ground truth — their label arrays are aligned with
    # the image features (clipped from frame 0), not the pose data.
    img_tcn_train_recs = [
        img_records_by_path[r.video_path]
        for r in train_records
        if r.video_path in img_manifest_idx
        and img_manifest_idx[r.video_path]["split"] == "train"
        and r.video_path in img_records_by_path
    ]
    img_tcn_train_probs  = [np.load(img_manifest_idx[r.video_path]["probs_path"]) for r in img_tcn_train_recs]
    img_tcn_train_labels = [p.argmax(axis=1).astype(np.int64) for p in img_tcn_train_probs]

    print(f"Loaded image TCN manifest — trial #{img_manifest['trial_id']}  "
          f"({len(img_tcn_train_recs)}/{len(train_records)} train records matched)")

    single_fold_img = [{
        "athlete":     "all_train",
        "val_records": img_tcn_train_recs,
        "raw_labels":  img_tcn_train_labels,
        "probs":       img_tcn_train_probs,
    }]

    img_tcn_results = {}
    for method, window in METHODS_CONFIG:
        needs_probs = method in PROB_METHODS
        preds = [
            apply_postprocessing(method, single_fold_img[0]["probs"][i] if needs_probs
                                 else single_fold_img[0]["raw_labels"][i], window)
            for i in range(len(img_tcn_train_recs))
        ]
        result = eval_fold_metrics(
            img_tcn_train_recs, preds, cfg.fps, cfg.n_classes, cfg.class_names
        ) | {"athlete": "all_train"}
        img_tcn_results[(method, window)] = [result]

    print(f"Computed {len(img_tcn_results)} (method, window) combinations for Image TCN.")

---
## Build summary table

In [ ]:
def build_summary(results_dict: dict, model_label: str) -> pd.DataFrame:
    rows = []
    for (method, window), folds in results_dict.items():
        for key in EVENT_KEYS:
            mae_vals    = [f["timing"][key]["ms"]        for f in folds
                           if not np.isnan(f["timing"][key]["ms"])]
            signed_vals = [f["timing"][key]["signed_ms"] for f in folds
                           if not np.isnan(f["timing"][key]["signed_ms"])]
            macro_f1s   = [f["f1"]["macro"] for f in folds]
            rows.append({
                "model":    model_label,
                "method":   method,
                "window":   window,
                "event":    key,
                "bias_ms":  np.mean(signed_vals) if signed_vals else np.nan,
                "mae_ms":   np.mean(mae_vals)    if mae_vals    else np.nan,
                "std_ms":   np.std(mae_vals)     if mae_vals    else np.nan,
                "macro_f1": np.mean(macro_f1s),
            })
    return pd.DataFrame(rows)


dfs = [build_summary(xgb_results, "XGBoost")]
if img_xgb_results is not None:
    dfs.append(build_summary(img_xgb_results, "Image XGBoost"))
if tcn_results is not None:
    dfs.append(build_summary(tcn_results, "Pose TCN"))
if img_tcn_results is not None:
    dfs.append(build_summary(img_tcn_results, "Image TCN"))

summary = pd.concat(dfs, ignore_index=True)

# Aggregate across events → one row per (model, method, window)
agg = (
    summary
    .groupby(["model", "method", "window"], sort=False)
    .agg(bias_ms=("bias_ms", "mean"),
         mae_ms= ("mae_ms",  "mean"),
         std_ms= ("std_ms",  "mean"),
         macro_f1=("macro_f1", "mean"))
    .reset_index()
)
agg["method_short"] = agg["method"].str.replace("_filter", "").str.replace("_argmax", "")
agg["label"] = agg["method_short"] + "  w=" + agg["window"].astype(str)

print(agg.to_string(index=False))

---
## Visualisation

In [ ]:
METHOD_COLORS = {
    "min_duration_filter":  "#e15759",
    "majority_vote_filter": "#4e79a7",
    "smooth_probs_argmax":  "#59a14f",
}
METHOD_LABELS = {
    "min_duration_filter":  "min_duration",
    "majority_vote_filter": "majority_vote",
    "smooth_probs_argmax":  "smooth_probs",
}

def _plot_metric(agg_df, models, metric, ylabel, title, axhline=None, ylim=None):
    n_models = len(models)
    fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 5), sharey=(ylim is not None))
    if n_models == 1:
        axes = [axes]

    for ax, model in zip(axes, models):
        sub = agg_df[agg_df["model"] == model].copy()
        for method, grp in sub.groupby("method", sort=False):
            grp = grp.sort_values("window")
            ax.plot(grp["window"], grp[metric], marker="o",
                    color=METHOD_COLORS[method], label=METHOD_LABELS[method])
        if axhline is not None:
            ax.axhline(axhline, color="gray", linestyle="--", linewidth=0.8)
        ax.set_xlabel("window size (frames)")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{model} — {title}")
        ax.set_xticks(WINDOWS)
        ax.legend()
        ax.grid(alpha=0.3)
        if ylim is not None:
            ax.set_ylim(*ylim)

    plt.tight_layout()
    plt.show()


models_present = agg["model"].unique().tolist()

_plot_metric(agg, models_present, "bias_ms",
             ylabel="mean signed error (ms)  [positive = late]",
             title="Timing bias",
             axhline=0)

In [ ]:
_plot_metric(agg, models_present, "mae_ms",
             ylabel="mean absolute error (ms)",
             title="Timing MAE")

In [ ]:
_plot_metric(agg, models_present, "macro_f1",
             ylabel="macro F1",
             title="Macro F1")

In [ ]:
# Per-event bias heatmap — rows = (method, window), cols = event
def plot_bias_heatmap(model_label):
    sub = summary[summary["model"] == model_label].copy()
    sub["config"] = (
        sub["method"].map(METHOD_LABELS) + "  w=" + sub["window"].astype(str)
    )
    pivot = sub.pivot_table(index="config", columns="event", values="bias_ms", aggfunc="mean")
    # Restore natural order
    ordered = [
        f"{METHOD_LABELS[m]}  w={w}"
        for m, w in METHODS_CONFIG
    ]
    pivot = pivot.reindex([c for c in ordered if c in pivot.index])

    fig, ax = plt.subplots(figsize=(9, len(pivot) * 0.45 + 1.5))
    vmax = np.nanmax(np.abs(pivot.values))
    sns.heatmap(pivot, annot=True, fmt=".1f", ax=ax,
                cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                linewidths=0.5, cbar_kws={"label": "signed error (ms)"})
    ax.set_title(f"{model_label} — per-event timing bias (ms) [positive = late]")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()


for m in models_present:
    plot_bias_heatmap(m)

---
## Best configuration per model

Rank by `|bias_ms|` (proximity to zero) as the primary criterion, then MAE as tiebreaker.

In [ ]:
agg["abs_bias"] = agg["bias_ms"].abs()

best = (
    agg
    .sort_values(["abs_bias", "mae_ms"])
    .groupby("model", sort=False)
    .first()
    [["method", "window", "bias_ms", "mae_ms", "macro_f1"]]
    .reset_index()
)
display(best)

print("\n--- copy these into compare_kinematic_xgboost.ipynb ---")
for _, row in best.iterrows():
    print(f"  {row['model']:10s}: method={row['method']!r:30s}  window={int(row['window'])}")